In [11]:
import pandas as pd
import os
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

Cargamos el archivo `corpus.csv` y visualizamos las primeras filas para entender su estructura.

In [12]:
ruta_input = "corpus.csv"

if os.path.exists(ruta_input):
    df = pd.read_csv(ruta_input, encoding='utf-8')
    print(f"Se cargaron {len(df)} películas.")
    display(df.head(3))
else:
    print(f"Error: No se encontró el archivo '{ruta_input}' en el directorio.")

Se cargaron 70 películas.


,titulo,idioma,sinopsis,reseña,género
0,Avatar,Español,Exploramos en Avatar la historia de una serie ...,"Tras verla, diría que es un poco lenta, pero c...","Acción, Aventura, Fantasía, Ciencia Ficción"
1,Pirates of the Caribbean: At World's End,Inglés,Acompañamos a los personajes de Pirates of the...,"Desde mi punto de vista, estamos ante impactan...","Aventura, Fantasía, Acción"
2,Spectre,Español,Acompañamos a los personajes de Spectre en un ...,Una experiencia cinematográfica fascinante y m...,"Acción, Aventura, Crimen"


Para cada película, unimos los siguientes campos en un único texto:
- `titulo` (Título de la película)
- `idioma` (Idioma de la película)
- `sinopsis` (Sinopsis de la película)
- `reseña` (Reseña de los usuarios)
- `género` (Géneros de la película)

Reemplazamos posibles valores nulos (`NaN`) con cadenas vacías para evitar errores de concatenación.
Además, aplicamos técnicas de normalización usando NLTK (conversión a minúsculas, tokenización, eliminación de signos de puntuación y palabras vacías, y reducción a la raíz o stemming).

In [13]:
# Configurar para español
stop_words = set(stopwords.words('spanish'))
stemmer = SnowballStemmer('spanish')

# Rellenar valores nulos con una cadena vacía
df = df.fillna("")

def elimina_no_alfanumerico(tokens):
    return [re.sub(r'[^\w]', '', token)
            for token in tokens
            if re.search(r'\w', token)]

def elimina_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

def aplica_stemmer(tokens):
    return [stemmer.stem(token) for token in tokens]

def procesar_texto(texto):
    texto = texto.lower()
    # Tokenizar
    tokens = word_tokenize(texto)
    tokens = elimina_no_alfanumerico(tokens)
    tokens = elimina_stopwords(tokens)
    tokens = aplica_stemmer(tokens)
    
    return " ".join(tokens)

# Crear una columna con el texto concatenado
df['documento_crudo'] = (
    df['titulo'] + 
    ". " + df['idioma'] + 
    " " + df['sinopsis'] + 
    " " + df['reseña'] + 
    " " + df['género']
)

# Crear la columna 'documento' procesada para el motor de búsqueda
df['documento'] = df['documento_crudo'].apply(procesar_texto)

In [14]:
print(f"Película: {df['titulo'].iloc[0]}\n")
print("Documento generado para búsqueda:")
print(df['documento'].iloc[0])

Película: Avatar

Documento generado para búsqueda:
avat español explor avat histori seri event misteri mantien suspens final ideal amant cin mensaj potent tras verl dir lent actuacion brillant recomend accion aventur fantas cienci ficcion


Guardamos el DataFrame resultante en un nuevo archivo CSV.

In [15]:
ruta_output = "corpus_procesado.csv"
df.to_csv(ruta_output, index=False, encoding='utf-8')
print(f" El corpus procesado se guardó en: {ruta_output}")

 El corpus procesado se guardó en: corpus_procesado.csv


### Creación del Índice Invertido con Whoosh
Utilizamos la biblioteca `Whoosh` para construir un índice invertido. Definiremos un esquema con el título y el contenido procesado, crearemos un directorio para el índice y añadiremos cada documento del corpus.

In [16]:
import os
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT, ID

# Definir el esquema del índice
schema = Schema(
    id=ID(stored=True, unique=True),
    titulo=TEXT(stored=True),
    idioma=TEXT(stored=True),
    sinopsis=TEXT(stored=True),
    reseña=TEXT(stored=True),
    genero=TEXT(stored=True),
    contenido=TEXT(stored=True) # Campo combinado procesado para búsqueda global
)

# Crear el directorio para el índice si no existe
index_dir = "indexdir"
if not os.path.exists(index_dir):
    os.mkdir(index_dir)

# Crear el índice
ix = create_in(index_dir, schema)

# Abrir un escritor para añadir documentos
writer = ix.writer()

# Iterar sobre el dataframe para añadir los documentos al índice
for i, row in df.iterrows():
    writer.add_document(
        id=str(i),
        titulo=str(row['titulo']),
        idioma=str(row['idioma']),
        sinopsis=str(row['sinopsis']),
        reseña=str(row['reseña']),
        genero=str(row['género']),
        contenido=str(row['documento'])
    )

writer.commit()
print("Índice invertido creado con éxito en el directorio 'indexdir'.")

Índice invertido creado con éxito en el directorio 'indexdir'.


### Sistema de Recuperación Booleano
A continuación, implementamos el sistema de recuperación booleano utilizando Whoosh. Se define una función para procesar la consulta del usuario (aplicando la misma normalización que a los documentos, pero preservando los operadores lógicos) y otra función para ejecutar la búsqueda y mostrar los resultados.

In [26]:
import re
from whoosh import qparser
from whoosh.index import open_dir

def procesar_query_booleana(query_str):
    """
    Procesa la consulta manteniendo los operadores lógicos booleanos intactos,
    pero aplicando el stemmer y eliminación de stopwords a los términos de búsqueda.
    """
    # Separar paréntesis para tratarlos como tokens independientes
    query_str = query_str.replace('(', ' ( ').replace(')', ' ) ')
    tokens = query_str.split()
    
    procesados = []
    for t in tokens:
        if t in ['AND', 'OR', 'NOT', '(', ')']:
            procesados.append(t)
        else:
            t_proc = procesar_texto(t)
            if t_proc:
                procesados.append(t_proc)
            
    return " ".join(procesados)

def buscar_booleano(query_str, index_dir="indexdir"):
    ix = open_dir(index_dir)
    
    query_procesada = procesar_query_booleana(query_str)
    print(f"Consulta original: {query_str}")
    print(f"Consulta procesada: {query_procesada}\n")
    
    with ix.searcher() as searcher:
        # Usamos el QueryParser en el campo 'contenido'
        parser = qparser.QueryParser("contenido", ix.schema)
        
        try:
            query = parser.parse(query_procesada)
            results = searcher.search(query, limit=None) # limit=None para devolver todos los compatibles
            
            print(f"Se encontraron {len(results)} documentos para la consulta.\n")
            for i, result in enumerate(results):
                print(f"--- Resultado {i+1} ---")
                print(f"Título: {result['titulo']}")
                print(f"Género: {result['genero']}")
                print(f"Idioma: {result['idioma']}")
                print("-" * 20)
                
        except Exception as e:
            print(f"Error al procesar la consulta: {e}")

# Ejecutamos una consulta de prueba
buscar_booleano("NOT Español AND Avatar OR (Pirates AND NOT Spectre)")

Consulta original: NOT Español AND Avatar OR (Pirates AND NOT Spectre)
Consulta procesada: NOT español AND avat OR ( pirat AND NOT spectr )

Se encontraron 3 documentos para la consulta.

--- Resultado 1 ---
Título: Pirates of the Caribbean: On Stranger Tides
Género: Aventura, Acción, Fantasía
Idioma: Inglés
--------------------
--- Resultado 2 ---
Título: Pirates of the Caribbean: At World's End
Género: Aventura, Fantasía, Acción
Idioma: Inglés
--------------------
--- Resultado 3 ---
Título: Pirates of the Caribbean: Dead Man's Chest
Género: Aventura, Fantasía, Acción
Idioma: Español
--------------------


### Interfaz Textual
Interfaz sencilla para realizar consultas booleanas interactivamente.

In [20]:
def interfaz_booleana():
    print("=== Sistema de Recuperación Booleano ===")
    print("Ingrese su consulta usando términos y operadores lógicos (AND, OR, NOT).")
    print("Escriba 'salir' para terminar.")
    
    while True:
        consulta = input("\nConsulta: ")
        if consulta.lower() == 'salir':
            print("Saliendo del sistema...")
            break
        if consulta.strip():
            buscar_booleano(consulta)

# Descomentar para usar la interfaz interactiva en Jupyter
interfaz_booleana()

=== Sistema de Recuperación Booleano ===
Ingrese su consulta usando términos y operadores lógicos (AND, OR, NOT).
Escriba 'salir' para terminar.
Consulta original: accion AND español
Consulta procesada: accion AND español

Se encontraron 25 documentos para la consulta.

--- Resultado 1 ---
Título: Superman Returns
Género: Aventura, Fantasía, Acción, Ciencia Ficción
Idioma: Español
--------------------
--- Resultado 2 ---
Título: Iron Man
Género: Acción, Ciencia Ficción, Aventura
Idioma: Español
--------------------
--- Resultado 3 ---
Título: X-Men: Days of Future Past
Género: Acción, Aventura, Fantasía, Ciencia Ficción
Idioma: Español
--------------------
--- Resultado 4 ---
Título: Transformers: Dark of the Moon
Género: Acción, Ciencia Ficción, Aventura
Idioma: Español
--------------------
--- Resultado 5 ---
Título: Spectre
Género: Acción, Aventura, Crimen
Idioma: Español
--------------------
--- Resultado 6 ---
Título: Furious 7
Género: Acción
Idioma: Español
--------------------
-